In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)
from sklearn.metrics import roc_curve, auc

In [ ]:
df = pd.read_csv("dataset/telco.csv")

print("Dataset Loaded Successfully")

In [ ]:
df.head()

In [ ]:
print("Rows and Columns:")
print(df.shape)

In [ ]:
print(df.columns)

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
df["Churn Label"].value_counts()

In [ ]:
df["Churn Label"].value_counts().plot(kind="bar")

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Count")

plt.show()

In [ ]:
df = df.drop(
    [
        "Customer ID",
        "Customer Status",
        "Churn Score",
        "Churn Category",
        "Churn Reason"
    ],
    axis=1
)

print("Columns Removed")

In [ ]:
df["Offer"] = df["Offer"].fillna("No Offer")
df["Internet Type"] = df["Internet Type"].fillna("No Internet")

print("Missing Values Handled")

In [ ]:
y = df["Churn Label"]

X = df.drop("Churn Label", axis=1)

print(X.shape)
print(y.shape)

In [ ]:
le_y = LabelEncoder()
y = le_y.fit_transform(y)

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in X.columns:
    if X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        encoders[col] = le

print("Encoding Completed")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

print("Decision Tree Trained")

In [ ]:
dt_predictions = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_predictions)

print("Decision Tree Accuracy:", dt_accuracy)

In [ ]:
cm = confusion_matrix(y_test, dt_predictions)

sns.heatmap(
    cm,
    annot=True,
    fmt="d"
)

plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
print(classification_report(y_test, dt_predictions))

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Random Forest Trained")

In [ ]:
rf_predictions = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_predictions)

print("Random Forest Accuracy:", rf_accuracy)

In [ ]:
rf_cm = confusion_matrix(y_test, rf_predictions)

sns.heatmap(
    rf_cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
print(classification_report(y_test, rf_predictions))

In [ ]:
from sklearn.metrics import f1_score

dt_f1 = f1_score(y_test, dt_predictions)
rf_f1 = f1_score(y_test, rf_predictions)

print("Decision Tree F1 Score :", dt_f1)
print("Random Forest F1 Score :", rf_f1)

In [ ]:
print("Decision Tree Accuracy :", dt_accuracy)
print("Random Forest Accuracy :", rf_accuracy)

print("\nDecision Tree F1 Score :", dt_f1)
print("Random Forest F1 Score :", rf_f1)

In [ ]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
)

feature_importance.nlargest(10).plot(kind="barh")

plt.title("Top 10 Important Features")

plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=['int64','float64'])

plt.figure(figsize=(12,8))
sns.heatmap(numeric_df.corr(), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
models = ["Decision Tree", "Random Forest"]
accuracies = [dt_accuracy, rf_accuracy]

plt.figure(figsize=(6,4))

bars = plt.bar(
    models,
    accuracies,
    color=["#81C784", "#64B5F6"]
)

plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)

# Values on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{height:.4f}",
        ha='center',
        va='bottom'
    )

plt.show()

In [ ]:
if rf_f1 > dt_f1:
    best_model = "Random Forest"
    best_f1 = rf_f1
else:
    best_model = "Decision Tree"
    best_f1 = dt_f1

print("Best Model Selected:", best_model)
print("Best F1 Score:", best_f1)

In [ ]:
# Decision Tree
dt_prob = dt_model.predict_proba(X_test)[:, 1]
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_prob)
roc_auc_dt = auc(fpr_dt, tpr_dt)

# Random Forest
rf_prob = rf_model.predict_proba(X_test)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)
roc_auc_rf = auc(fpr_rf, tpr_rf)

# Plot comparison
plt.figure(figsize=(8,6))

plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree AUC = {roc_auc_dt:.4f}")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest AUC = {roc_auc_rf:.4f}")

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()

plt.show()

print("Decision Tree AUC:", roc_auc_dt)
print("Random Forest AUC:", roc_auc_rf)

In [ ]:

comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": rf_predictions
})

comparison.head(10)

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": rf_predictions
})

comparison["Actual"] = comparison["Actual"].map({
    0: "No",
    1: "Yes"
})

comparison["Predicted"] = comparison["Predicted"].map({
    0: "No",
    1: "Yes"
})

comparison.head(10)

In [ ]:
correct_predictions = (
    comparison["Actual"]
    ==
    comparison["Predicted"]
).sum()

print("Correct Predictions:", correct_predictions)

print("Total Test Samples:", len(comparison))

In [ ]:
df["Churn Label"].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%"
)

plt.title("Customer Churn Percentage")
plt.ylabel("")
plt.show()

In [ ]:
pd.crosstab(
    df["Contract"],
    df["Churn Label"]
)

In [ ]:
import joblib

joblib.dump(
    rf_model,
    "best_churn_model.pkl"
)

print("Model Saved Successfully")

# FINAL DEPLOYMENT MODEL (Top Features)

In [ ]:
from sklearn.preprocessing import LabelEncoder

y = df["Churn Label"].copy()

le_y = LabelEncoder()
y = le_y.fit_transform(y)

In [ ]:
top_features = [
    "Satisfaction Score",
    "Contract",
    "Monthly Charge",
    "Tenure in Months",
    "Total Revenue",
    "Number of Referrals",
    "Total Charges",
    "Total Long Distance Charges",
    "Age"
]

print("Deployment Features:")
print(top_features)

In [ ]:
print(df.columns.tolist())

In [ ]:
missing = [col for col in top_features if col not in df.columns]
print("Missing columns:", missing)

In [ ]:
X_reduced = df[top_features].copy()

In [ ]:
# Ensure consistent encoding by using the same Contract encoder from training
X_reduced["Contract"] = encoders["Contract"].transform(X_reduced["Contract"])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

final_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

final_model.fit(X_train, y_train)

print("Final Model Trained using Selected Features")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = final_model.predict(X_test)

print("Final Model Accuracy:", accuracy_score(y_test, y_pred))
print("Final Model F1 Score:", f1_score(y_test, y_pred))

In [ ]:

joblib.dump(encoders, "encoders.pkl")

In [ ]:
import joblib

joblib.dump(final_model, "churn_model_streamlit.pkl")

print("Streamlit Model Saved Successfully")

# Final Conclusion

Two machine learning models were developed and evaluated for customer churn prediction.

## Dataset Used:
Telco Customer Churn Dataset

## Models Used:
1. Decision Tree Classifier  
2. Random Forest Classifier  

## Evaluation Metrics Used:
Accuracy, F1 Score, Confusion Matrix, ROC-AUC

## Results:
- Decision Tree Accuracy: 94.04%  
- Random Forest Accuracy: 95.32%  

## Best Model Selected:
**Random Forest Classifier**

## Saved Models:
- `best_churn_model.pkl` → Full feature Random Forest model  
- `churn_model_streamlit.pkl` → Optimized deployment model with selected features  

## Final Outcome:
The Random Forest model achieved the highest accuracy and was selected as the final model for customer churn prediction.

Feature selection helped reduce model complexity while maintaining strong predictive performance.

## Project Status:
The customer churn prediction system is successfully completed and ready for real-time use.